# 01 - Exploratory Data Analysis
**Project:** Credit Card Default Prediction  
**Dataset:** UCI Default of Credit Card Clients (30,000 records, 23 features)

---
### Environment Setup
- **VS Code / Local:** Data path is `../data/UCI_Credit_Card.csv` (default, no changes needed)
- **Google Colab:** Upload the CSV manually, then change `DATA_PATH` to `'UCI_Credit_Card.csv'`

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# Change to 'UCI_Credit_Card.csv' if using Google Colab
DATA_PATH = '../data/UCI_Credit_Card.csv'

df = pd.read_csv(DATA_PATH)

# Rename target column for easier access
df.rename(columns={'default payment next month': 'default'}, inplace=True)

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ── Basic Info ────────────────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Summary Statistics ===')
df.describe().T

In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
# Check target imbalance: 0 = No default, 1 = Default
default_counts = df['default'].value_counts()
default_pct = df['default'].value_counts(normalize=True) * 100

print('=== Target Distribution ===')
print(pd.DataFrame({'Count': default_counts, 'Percentage (%)': default_pct.round(2)}))

fig, ax = plt.subplots()
default_counts.plot(kind='bar', color=['steelblue', 'salmon'], ax=ax)
ax.set_title('Target Class Distribution')
ax.set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('\nNote: ~22% default rate means the dataset is imbalanced.')
print('Accuracy alone will be misleading — we will prioritize AUC-ROC and Recall.')

In [ ]:
# ── Default Rate by Demographic Features ──────────────────────────────────────
# SEX: 1=Male, 2=Female
# EDUCATION: 1=Graduate school, 2=University, 3=High school, 4=Others
# MARRIAGE: 1=Married, 2=Single, 3=Others

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title in zip(axes,
                           ['SEX', 'EDUCATION', 'MARRIAGE'],
                           ['Gender (1=Male, 2=Female)', 'Education Level', 'Marital Status']):
    df.groupby(col)['default'].mean().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Default Rate by {title}')
    ax.set_ylabel('Default Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('Default Rate by Demographic Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Credit Limit vs Default ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of credit limit
axes[0].hist(df['LIMIT_BAL'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Credit Limit Distribution')
axes[0].set_xlabel('Credit Limit (NT$)')
axes[0].set_ylabel('Count')

# Credit limit by default status
df.boxplot(column='LIMIT_BAL', by='default', ax=axes[1])
axes[1].set_title('Credit Limit by Default Status')
axes[1].set_xlabel('Default (0=No, 1=Yes)')
axes[1].set_ylabel('Credit Limit (NT$)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('Observation: Customers who defaulted tend to have lower credit limits.')

In [ ]:
# ── Payment Status Features (PAY_0 to PAY_6) ─────────────────────────────────
# PAY_0 = repayment status in September 2005 (most recent)
# Values: -1=pay duly, 1=1 month delay, 2=2 months delay, etc.
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, pay_cols):
    df.groupby(col)['default'].mean().plot(kind='bar', ax=ax, color='salmon')
    ax.set_title(f'Default Rate by {col}')
    ax.set_ylabel('Default Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('Default Rate by Payment Status (Last 6 Months)', fontsize=13)
plt.tight_layout()
plt.show()

print('Observation: Higher payment delay codes strongly correlate with default.')

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
# Focus on correlation between numeric features and the target
corr = df.corr()[['default']].sort_values('default', ascending=False)

plt.figure(figsize=(6, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation with Default (Target Variable)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Key Takeaways ─────────────────────────────────────────────────────────────
print('=== EDA Summary ===')
print('1. Dataset has no missing values — no imputation needed.')
print('2. Target is imbalanced (~22% default) — use AUC-ROC and Recall as primary metrics.')
print('3. Payment status features (PAY_0 to PAY_6) are the strongest predictors of default.')
print('4. Lower credit limits are associated with higher default risk.')
print('5. Demographic features (gender, education, marital status) show minor differences in default rate.')